## **Step 1 — Load R1 data**

In [29]:
from pathlib import Path

print("Current working directory:")
print(Path.cwd())

release_dir = Path("../data/robustness/releases/R1")

print("\nExpected R1 directory:")
print(release_dir.resolve())

print("\nDirectory exists:", release_dir.exists())

if release_dir.exists():
    print("\nCSV files:")
    for file in sorted(release_dir.glob("*.csv")):
        print(file.name)

Current working directory:
<PROJECT_ROOT>\notebooks

Expected R1 directory:
<PROJECT_ROOT>\data\robustness\releases\R1

Directory exists: True

CSV files:
account_month_features.csv
accounts.csv
constant_history_departures.csv
eligibility_summary.csv
evaluation_labels.csv
export_manifest.csv
export_schema.csv
historical_assessments.csv
historical_comparison.csv
historical_coverage.csv
historical_diagnostics.csv
historical_method_summary.csv
injection_eligibility.csv
injection_manifest.csv
injection_operations.csv
integrity_checks.csv
protected_files.csv
run_metadata.csv
scenario_invariants.csv
selected_allocation.csv
transactions.csv


In [30]:
from pathlib import Path
import pandas as pd
import numpy as np

release_dir = Path("../data/robustness/releases/R1")

features_df = pd.read_csv(
    release_dir / "account_month_features.csv"
)

labels_df = pd.read_csv(
    release_dir / "evaluation_labels.csv"
)

print("Features shape:", features_df.shape)
print("Labels shape:", labels_df.shape)

display(labels_df.head())

Features shape: (6000, 22)
Labels shape: (6000, 4)


,account_id,month,is_injected_anomaly,scenario
0,ACC00001,1,0,normal
1,ACC00001,2,0,normal
2,ACC00001,3,0,normal
3,ACC00001,4,1,S5_combined_moderate_shift
4,ACC00001,5,0,normal


## **Validate R1 labels**

In [31]:
print("Features columns:")
print(features_df.columns.tolist())

print("\nLabels columns:")
print(labels_df.columns.tolist())

print("\nFirst five labels:")
display(labels_df.head())

Features columns:
['account_id', 'month', 'txn_count', 'unique_counterparties', 'inflow_sum', 'outflow_sum', 'inflow_avg', 'outflow_avg', 'net_flow', 'inflow_outflow_ratio', 'zero_outflow_flag', 'flow_concentration', 'txn_count_change_vs_prev', 'txn_count_robust_z_past', 'unique_counterparties_change_vs_prev', 'unique_counterparties_robust_z_past', 'inflow_sum_change_vs_prev', 'inflow_sum_robust_z_past', 'outflow_sum_change_vs_prev', 'outflow_sum_robust_z_past', 'net_flow_change_vs_prev', 'net_flow_robust_z_past']

Labels columns:
['account_id', 'month', 'is_injected_anomaly', 'scenario']

First five labels:


,account_id,month,is_injected_anomaly,scenario
0,ACC00001,1,0,normal
1,ACC00001,2,0,normal
2,ACC00001,3,0,normal
3,ACC00001,4,1,S5_combined_moderate_shift
4,ACC00001,5,0,normal


## **Step 3 — Validate R1 and merge features with labels**

In [32]:
# Check dataset sizes
assert len(features_df) == 6000
assert len(labels_df) == 6000

# Check unique account-month keys
assert not features_df.duplicated(
    ["account_id", "month"]
).any()

assert not labels_df.duplicated(
    ["account_id", "month"]
).any()

# Check anomaly distribution
anomalies_by_month = (
    labels_df.groupby("month")["is_injected_anomaly"]
    .sum()
    .reindex(range(1, 7), fill_value=0)
)

print("Anomalies by month:")
print(anomalies_by_month)

assert anomalies_by_month.tolist() == [0, 0, 0, 50, 50, 50]

# Merge features and labels
df = features_df.merge(
    labels_df,
    on=["account_id", "month"],
    how="inner",
    validate="one_to_one"
)

assert len(df) == 6000

print("\nMerged dataset shape:", df.shape)
print("R1 validation passed!")

Anomalies by month:
month
1     0
2     0
3     0
4    50
5    50
6    50
Name: is_injected_anomaly, dtype: int64

Merged dataset shape: (6000, 24)
R1 validation passed!


# **Step 4 — Chronological Train/Validation/Test Split**

In [33]:
# Chronological split
train_df = df[df["month"].isin([1, 2, 3])].copy()
val_df   = df[df["month"] == 4].copy()
test_df  = df[df["month"].isin([5, 6])].copy()

# Verify split sizes
assert len(train_df) == 3000
assert len(val_df) == 1000
assert len(test_df) == 2000

# Verify anomaly counts
assert train_df["is_injected_anomaly"].sum() == 0
assert val_df["is_injected_anomaly"].sum() == 50
assert test_df["is_injected_anomaly"].sum() == 100

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

print("\nAnomaly counts:")
print("Train:", train_df["is_injected_anomaly"].sum())
print("Validation:", val_df["is_injected_anomaly"].sum())
print("Test:", test_df["is_injected_anomaly"].sum())

Train: (3000, 24)
Validation: (1000, 24)
Test: (2000, 24)

Anomaly counts:
Train: 0
Validation: 50
Test: 100


# **Step 5 — Feature Selection**

**Step 5.1 — Define the four feature groups**

In [34]:
behavioral_features = [
    "txn_count",
    "unique_counterparties",
    "txn_count_change_vs_prev",
    "unique_counterparties_change_vs_prev"
]

fund_flow_core_features = [
    "inflow_sum",
    "outflow_sum",
    "inflow_avg",
    "outflow_avg",
    "net_flow",
    "inflow_outflow_ratio",
    "flow_concentration"
]

combined_core_features = (
    behavioral_features + fund_flow_core_features
)

fund_flow_extended_features = [
    "inflow_sum",
    "outflow_sum",
    "inflow_avg",
    "outflow_avg",
    "net_flow",
    "inflow_outflow_ratio",
    "zero_outflow_flag",
    "flow_concentration",
    "inflow_sum_change_vs_prev",
    "outflow_sum_change_vs_prev",
    "net_flow_change_vs_prev"
]

feature_groups = {
    "Behavioral": behavioral_features,
    "Fund Flow Core": fund_flow_core_features,
    "Combined Core": combined_core_features,
    "Fund Flow Extended": fund_flow_extended_features
}

**Step 5.2 — Validate feature availability**

In [35]:
for model_name, columns in feature_groups.items():

    missing = set(columns) - set(features_df.columns)

    assert not missing, (
        f"{model_name} has missing features: {missing}"
    )

    assert len(columns) == len(set(columns)), (
        f"{model_name} contains duplicate features"
    )

print("All feature groups validated!")

All feature groups validated!


**Step 5.3 — Prepare labels separately**

In [36]:
train_features = features_df[
    features_df["month"].isin([1, 2, 3])
].copy()

val_features = features_df[
    features_df["month"] == 4
].copy()

test_features = features_df[
    features_df["month"].isin([5, 6])
].copy()

train_labels = labels_df[
    labels_df["month"].isin([1, 2, 3])
].copy()

val_labels = labels_df[
    labels_df["month"] == 4
].copy()

test_labels = labels_df[
    labels_df["month"].isin([5, 6])
].copy()

# Explicitly align labels to feature rows by account-month keys
def aligned_targets(feature_rows, label_rows):
    aligned = feature_rows[
        ["account_id", "month"]
    ].merge(
        label_rows[
            ["account_id", "month", "is_injected_anomaly"]
        ],
        on=["account_id", "month"],
        how="left",
        validate="one_to_one"
    )

    assert aligned["is_injected_anomaly"].notna().all()

    return aligned["is_injected_anomaly"].to_numpy(dtype=int)

y_train = aligned_targets(train_features, train_labels)
y_val = aligned_targets(val_features, val_labels)
y_test = aligned_targets(test_features, test_labels)

print("Train:", len(y_train), "anomalies:", y_train.sum())
print("Validation:", len(y_val), "anomalies:", y_val.sum())
print("Test:", len(y_test), "anomalies:", y_test.sum())

Train: 3000 anomalies: 0
Validation: 1000 anomalies: 50
Test: 2000 anomalies: 100


# **Step 6 — Median Imputation**

**Step 6.1 — Inspect missing values**

In [37]:
for name, columns in feature_groups.items():
    print(f"\n{name}")

    print(
        train_features[columns]
        .isna()
        .sum()
        .loc[lambda x: x > 0]
    )


Behavioral
txn_count_change_vs_prev                1000
unique_counterparties_change_vs_prev    1000
dtype: int64

Fund Flow Core
inflow_avg              17
outflow_avg             13
inflow_outflow_ratio    13
dtype: int64

Combined Core
txn_count_change_vs_prev                1000
unique_counterparties_change_vs_prev    1000
inflow_avg                                17
outflow_avg                               13
inflow_outflow_ratio                      13
dtype: int64

Fund Flow Extended
inflow_avg                      17
outflow_avg                     13
inflow_outflow_ratio            13
inflow_sum_change_vs_prev     1011
outflow_sum_change_vs_prev    1008
net_flow_change_vs_prev       1000
dtype: int64


**Step 6.2 — Create an imputation function**

In [38]:
def prepare_model_features(
    train_df,
    val_df,
    test_df,
    columns
):
    # Select only the model's features
    X_train = train_df[columns].copy()
    X_val = val_df[columns].copy()
    X_test = test_df[columns].copy()

    # Calculate medians using TRAIN ONLY
    train_medians = X_train.median()

    # Guard against features with no observed training values
    if train_medians.isna().any():
        missing = train_medians[
            train_medians.isna()
        ].index.tolist()

        raise ValueError(
            f"All-missing training features: {missing}"
        )

    # Apply frozen training medians
    X_train = X_train.fillna(train_medians)
    X_val = X_val.fillna(train_medians)
    X_test = X_test.fillna(train_medians)

    # Validate preprocessing
    assert not X_train.isna().any().any()
    assert not X_val.isna().any().any()
    assert not X_test.isna().any().any()

    return X_train, X_val, X_test, train_medians

**Step 6.3 — Prepare all four models**

In [39]:
prepared_data = {}

for model_name, columns in feature_groups.items():

    X_train, X_val, X_test, medians = (
        prepare_model_features(
            train_features,
            val_features,
            test_features,
            columns
        )
    )

    prepared_data[model_name] = {
        "X_train": X_train,
        "X_val": X_val,
        "X_test": X_test,
        "medians": medians
    }

    print(
        f"{model_name}: "
        f"Train={X_train.shape}, "
        f"Validation={X_val.shape}, "
        f"Test={X_test.shape}"
    )

Behavioral: Train=(3000, 4), Validation=(1000, 4), Test=(2000, 4)
Fund Flow Core: Train=(3000, 7), Validation=(1000, 7), Test=(2000, 7)
Combined Core: Train=(3000, 11), Validation=(1000, 11), Test=(2000, 11)
Fund Flow Extended: Train=(3000, 11), Validation=(1000, 11), Test=(2000, 11)


# **Step 7 — Train the four Isolation Forest models**

In [40]:
IF_SETTINGS = {
    "random_state": 42,
    "n_estimators": 200,
    "max_samples": "auto",
    "contamination": "auto"
}

**Step 7.1 — Train and calculate anomaly scores**

In [41]:
from sklearn.ensemble import IsolationForest

models = {}
scores = {}

for model_name, data in prepared_data.items():

    model = IsolationForest(**IF_SETTINGS)

    model.fit(data["X_train"])

    models[model_name] = model

    scores[model_name] = {
        "train": -model.score_samples(data["X_train"]),
        "val": -model.score_samples(data["X_val"]),
        "test": -model.score_samples(data["X_test"])
    }

    print(
        f"{model_name}: "
        f"trained successfully | "
        f"Validation scores={len(scores[model_name]['val'])} | "
        f"Test scores={len(scores[model_name]['test'])}"
    )

Behavioral: trained successfully | Validation scores=1000 | Test scores=2000
Fund Flow Core: trained successfully | Validation scores=1000 | Test scores=2000
Combined Core: trained successfully | Validation scores=1000 | Test scores=2000
Fund Flow Extended: trained successfully | Validation scores=1000 | Test scores=2000


**Step 7.2 — Verify score distributions**

In [42]:
for model_name, model_scores in scores.items():

    print(f"\n{model_name}")

    for split in ["train", "val", "test"]:
        values = model_scores[split]

        print(
            f"{split}: "
            f"min={values.min():.4f}, "
            f"mean={values.mean():.4f}, "
            f"max={values.max():.4f}"
        )


Behavioral
train: min=0.3536, mean=0.4331, max=0.7900
val: min=0.3552, mean=0.4574, max=0.7707
test: min=0.3536, mean=0.4566, max=0.7976

Fund Flow Core
train: min=0.3565, mean=0.4344, max=0.7286
val: min=0.3576, mean=0.4383, max=0.7260
test: min=0.3573, mean=0.4370, max=0.7376

Combined Core
train: min=0.3493, mean=0.4263, max=0.6995
val: min=0.3543, mean=0.4378, max=0.6553
test: min=0.3542, mean=0.4368, max=0.6909

Fund Flow Extended
train: min=0.3390, mean=0.4036, max=0.6781
val: min=0.3427, mean=0.4148, max=0.7262
test: min=0.3410, mean=0.4151, max=0.7317


# **Step 8 — Validation threshold calibration**

**Step 8.1 — Define the threshold selection function**

In [43]:
from sklearn.metrics import precision_score, recall_score, f1_score
import numpy as np

def select_best_threshold(y_true, anomaly_scores):

    thresholds = np.append(
        np.unique(anomaly_scores),
        np.nextafter(np.max(anomaly_scores), np.inf)
    )
    results = []

    for threshold in thresholds:

        y_pred = (
            anomaly_scores >= threshold
        ).astype(int)

        precision = precision_score(
            y_true, y_pred, zero_division=0
        )

        recall = recall_score(
            y_true, y_pred, zero_division=0
        )

        f1 = f1_score(
            y_true, y_pred, zero_division=0
        )

        fp = int(((y_true == 0) & (y_pred == 1)).sum())

        results.append({
            "threshold": float(threshold),
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "fp": fp
        })

    best = sorted(
        results,
        key=lambda r: (
            -r["f1"],
            r["fp"],
            -r["threshold"]
        )
    )[0]

    return best

**Step 8.2 — Calibrate all four models**

In [44]:
threshold_results = {}

for model_name in feature_groups:

    result = select_best_threshold(
        y_val,
        scores[model_name]["val"]
    )

    threshold_results[model_name] = result

    print(f"\n{model_name}")
    print(f"Threshold: {result['threshold']:.6f}")
    print(f"Precision: {result['precision']:.4f}")
    print(f"Recall:    {result['recall']:.4f}")
    print(f"F1:        {result['f1']:.4f}")
    print(f"FP:        {result['fp']}")


Behavioral
Threshold: 0.573441
Precision: 0.1828
Recall:    0.3400
F1:        0.2378
FP:        76

Fund Flow Core
Threshold: 0.586569
Precision: 0.2917
Recall:    0.2800
F1:        0.2857
FP:        34

Combined Core
Threshold: 0.534503
Precision: 0.2308
Recall:    0.4200
F1:        0.2979
FP:        70

Fund Flow Extended
Threshold: 0.542708
Precision: 0.3400
Recall:    0.3400
F1:        0.3400
FP:        33


## **Step 9 — Evaluate R1 on the test dataset**

**Step 9.1 — Calculate test metrics**

In [45]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    confusion_matrix
)

test_results = {}

for model_name in feature_groups:

    threshold = threshold_results[model_name]["threshold"]
    test_scores = scores[model_name]["test"]

    y_pred = (test_scores >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_test, y_pred, labels=[0, 1]
    ).ravel()

    test_results[model_name] = {
        "threshold": threshold,
        "TP": int(tp),
        "FP": int(fp),
        "FN": int(fn),
        "TN": int(tn),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "AP": average_precision_score(y_test, test_scores),
        "alerts": int(y_pred.sum())
    }

results_df = pd.DataFrame.from_dict(
    test_results, orient="index"
)

display(results_df.round(4))

,threshold,TP,FP,FN,TN,precision,recall,f1,AP,alerts
Behavioral,0.5734,26,152,74,1748,0.1461,0.26,0.1871,0.1097,178
Fund Flow Core,0.5866,25,66,75,1834,0.2747,0.25,0.2618,0.2140,91
Combined Core,0.5345,37,137,63,1763,0.2126,0.37,0.2701,0.1944,174
Fund Flow Extended,0.5427,26,76,74,1824,0.2549,0.26,0.2574,0.2694,102


**Step 9.2 — Precision@100 and Recall@100**

In [46]:
K = 100

for model_name in feature_groups:

    test_scores = scores[model_name]["test"]

    # Stable ranking for reproducibility
    top_indices = np.argsort(
        -test_scores, kind="stable"
    )[:K]

    top_true_positives = int(y_test[top_indices].sum())

    precision_at_100 = top_true_positives / K
    recall_at_100 = top_true_positives / int(y_test.sum())

    test_results[model_name]["Precision@100"] = precision_at_100
    test_results[model_name]["Recall@100"] = recall_at_100

results_df = pd.DataFrame.from_dict(
    test_results, orient="index"
)

display(results_df.round(4))

,threshold,TP,FP,FN,TN,precision,recall,f1,AP,alerts,Precision@100,Recall@100
Behavioral,0.5734,26,152,74,1748,0.1461,0.26,0.1871,0.1097,178,0.16,0.16
Fund Flow Core,0.5866,25,66,75,1834,0.2747,0.25,0.2618,0.2140,91,0.27,0.27
Combined Core,0.5345,37,137,63,1763,0.2126,0.37,0.2701,0.1944,174,0.26,0.26
Fund Flow Extended,0.5427,26,76,74,1824,0.2549,0.26,0.2574,0.2694,102,0.26,0.26


# **Step 10 — Prepare R1 Fusion**

In [47]:
fusion_sources = ["Behavioral", "Fund Flow Core"]

standardized_scores = {}

for model_name in fusion_sources:

    train_scores = scores[model_name]["train"]

    train_mean = np.mean(train_scores)
    train_std = np.std(train_scores, ddof=0)

    assert train_std > 0

    standardized_scores[model_name] = {}

    for split in ["train", "val", "test"]:

        standardized_scores[model_name][split] = (
            scores[model_name][split] - train_mean
        ) / train_std

    print(
        f"{model_name}: "
        f"Train mean={train_mean:.6f}, "
        f"Train std={train_std:.6f}"
    )

Behavioral: Train mean=0.433078, Train std=0.069889
Fund Flow Core: Train mean=0.434430, Train std=0.066276


**Step 10.2 — Calculate Mean and Max Fusion scores**

In [48]:
fusion_scores = {
    "Mean Fusion": {},
    "Max Fusion": {}
}

for split in ["val", "test"]:

    behavioral = standardized_scores["Behavioral"][split]
    fund_flow = standardized_scores["Fund Flow Core"][split]

    fusion_scores["Mean Fusion"][split] = (
        behavioral + fund_flow
    ) / 2

    fusion_scores["Max Fusion"][split] = np.maximum(
        behavioral, fund_flow
    )

for fusion_name in fusion_scores:

    print(f"\n{fusion_name}")

    for split in ["val", "test"]:
        values = fusion_scores[fusion_name][split]

        print(
            f"{split}: "
            f"min={values.min():.4f}, "
            f"mean={values.mean():.4f}, "
            f"max={values.max():.4f}"
        )


Mean Fusion
val: min=-1.0380, mean=0.2035, max=3.5093
test: min=-1.1059, mean=0.1881, max=3.7987

Max Fusion
val: min=-0.9654, mean=0.7538, max=4.8305
test: min=-1.0989, mean=0.7539, max=5.2157


# **Step 11 — Calibrate Fusion thresholds**

In [49]:
fusion_threshold_results = {}

for fusion_name in ["Mean Fusion", "Max Fusion"]:

    result = select_best_threshold(
        y_val,
        fusion_scores[fusion_name]["val"]
    )

    fusion_threshold_results[fusion_name] = result

    print(f"\n{fusion_name}")
    print(f"Threshold: {result['threshold']:.6f}")
    print(f"Precision: {result['precision']:.4f}")
    print(f"Recall:    {result['recall']:.4f}")
    print(f"F1:        {result['f1']:.4f}")
    print(f"FP:        {result['fp']}")


Mean Fusion
Threshold: 1.407948
Precision: 0.2130
Recall:    0.4600
F1:        0.2911
FP:        85

Max Fusion
Threshold: 2.339084
Precision: 0.2190
Recall:    0.4600
F1:        0.2968
FP:        82


**Step 12 — Evaluate Union, Mean and Max Fusion**

In [50]:
from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score
)

fusion_test_results = {}

for fusion_name in ["Union", "Mean Fusion", "Max Fusion"]:

    if fusion_name == "Union":

        behavioral_pred = (
            scores["Behavioral"]["test"] >=
            threshold_results["Behavioral"]["threshold"]
        )

        fund_flow_pred = (
            scores["Fund Flow Core"]["test"] >=
            threshold_results["Fund Flow Core"]["threshold"]
        )

        y_pred = (
            behavioral_pred | fund_flow_pred
        ).astype(int)

        threshold = None
        ap = np.nan

    else:

        threshold = fusion_threshold_results[
            fusion_name
        ]["threshold"]

        test_scores = fusion_scores[fusion_name]["test"]

        y_pred = (test_scores >= threshold).astype(int)

        ap = average_precision_score(
            y_test, test_scores
        )

    tn, fp, fn, tp = confusion_matrix(
        y_test, y_pred, labels=[0, 1]
    ).ravel()

    fusion_test_results[fusion_name] = {
        "threshold": threshold,
        "TP": int(tp),
        "FP": int(fp),
        "FN": int(fn),
        "TN": int(tn),
        "precision": precision_score(
            y_test, y_pred, zero_division=0
        ),
        "recall": recall_score(
            y_test, y_pred, zero_division=0
        ),
        "f1": f1_score(
            y_test, y_pred, zero_division=0
        ),
        "AP": ap,
        "alerts": int(y_pred.sum())
    }

fusion_results_df = pd.DataFrame.from_dict(
    fusion_test_results,
    orient="index"
)

display(fusion_results_df.round(4))

,threshold,TP,FP,FN,TN,precision,recall,f1,AP,alerts
Union,NaN,45,207,55,1693,0.1786,0.45,0.2557,NaN,252
Mean Fusion,1.4079,35,146,65,1754,0.1934,0.35,0.2491,0.1666,181
Max Fusion,2.3391,41,162,59,1738,0.2020,0.41,0.2706,0.1680,203


## **Step 13 — Compare Combined Core and Max Fusion**

In [51]:
comparison = pd.DataFrame({
    "Combined Core": test_results["Combined Core"],
    "Max Fusion": fusion_test_results["Max Fusion"]
}).T

display(
    comparison[
        ["TP", "FP", "precision", "recall",
         "f1", "AP", "alerts"]
    ].round(4)
)

,TP,FP,precision,recall,f1,AP,alerts
Combined Core,37.0,137.0,0.2126,0.37,0.2701,0.1944,174.0
Max Fusion,41.0,162.0,0.2020,0.41,0.2706,0.1680,203.0


## **Step 14 — Precision@100 and Recall@100 for Fusion**

In [52]:
K = 100

for fusion_name in ["Mean Fusion", "Max Fusion"]:

    test_scores = fusion_scores[fusion_name]["test"]

    top_indices = np.argsort(
        -test_scores, kind="stable"
    )[:K]

    tp_at_100 = int(y_test[top_indices].sum())

    precision_at_100 = tp_at_100 / K
    recall_at_100 = tp_at_100 / int(y_test.sum())

    fusion_test_results[fusion_name]["Precision@100"] = (
        precision_at_100
    )

    fusion_test_results[fusion_name]["Recall@100"] = (
        recall_at_100
    )

    print(f"\n{fusion_name}")
    print(f"TP@100: {tp_at_100}")
    print(f"Precision@100: {precision_at_100:.4f}")
    print(f"Recall@100: {recall_at_100:.4f}")


Mean Fusion
TP@100: 26
Precision@100: 0.2600
Recall@100: 0.2600

Max Fusion
TP@100: 20
Precision@100: 0.2000
Recall@100: 0.2000


## **Step 15 — Save the complete R1 results**

In [53]:
# Combine model and fusion results
r1_results = pd.concat([
    pd.DataFrame.from_dict(
        test_results, orient="index"
    ),
    pd.DataFrame.from_dict(
        fusion_test_results, orient="index"
    )
])

r1_results.index.name = "model"
r1_results.insert(0, "replication", "R1")

# Save results
output_dir = Path("../results/robustness")
output_dir.mkdir(parents=True, exist_ok=True)

output_file = output_dir / "R1_results.csv"

r1_results.to_csv(output_file)

print("Saved:", output_file.resolve())

display(r1_results.round(4))

Saved: <PROJECT_ROOT>\results\robustness\R1_results.csv


,replication,threshold,TP,FP,FN,TN,precision,recall,f1,AP,alerts,Precision@100,Recall@100
model,,,,,,,,,,,,,
Behavioral,R1,0.5734,26,152,74,1748,0.1461,0.26,0.1871,0.1097,178,0.16,0.16
Fund Flow Core,R1,0.5866,25,66,75,1834,0.2747,0.25,0.2618,0.2140,91,0.27,0.27
Combined Core,R1,0.5345,37,137,63,1763,0.2126,0.37,0.2701,0.1944,174,0.26,0.26
Fund Flow Extended,R1,0.5427,26,76,74,1824,0.2549,0.26,0.2574,0.2694,102,0.26,0.26
Union,R1,NaN,45,207,55,1693,0.1786,0.45,0.2557,NaN,252,NaN,NaN
Mean Fusion,R1,1.4079,35,146,65,1754,0.1934,0.35,0.2491,0.1666,181,0.26,0.26
Max Fusion,R1,2.3391,41,162,59,1738,0.2020,0.41,0.2706,0.1680,203,0.20,0.20


## **Step 16 — Save the R1 validation thresholds**

In [54]:
validation_records = []

for model_name, result in threshold_results.items():
    validation_records.append({
        "replication": "R1",
        "model": model_name,
        **result
    })

for model_name, result in fusion_threshold_results.items():
    validation_records.append({
        "replication": "R1",
        "model": model_name,
        **result
    })

validation_df = pd.DataFrame(validation_records)

validation_file = output_dir / "R1_validation.csv"

validation_df.to_csv(validation_file, index=False)

print("Saved:", validation_file.resolve())

display(validation_df.round(4))

Saved: <PROJECT_ROOT>\results\robustness\R1_validation.csv


,replication,model,threshold,precision,recall,f1,fp
0,R1,Behavioral,0.5734,0.1828,0.34,0.2378,76
1,R1,Fund Flow Core,0.5866,0.2917,0.28,0.2857,34
2,R1,Combined Core,0.5345,0.2308,0.42,0.2979,70
3,R1,Fund Flow Extended,0.5427,0.3400,0.34,0.3400,33
4,R1,Mean Fusion,1.4079,0.2130,0.46,0.2911,85
5,R1,Max Fusion,2.3391,0.2190,0.46,0.2968,82


## **Step 17 — Check the exact feature definitions**